In [ ]:
pip install together


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.4/86.4 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.3/120.3 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 1.6 MB/s eta 0:00:00
  Attempting uninstall: typer
    Found existing installation: typer 0.20.0
    Uninstalling typer-0.20.0:
      Successfully uninstalled typer-0.20.0


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Code to run experiments on thinking models

In [ ]:
import pickle
import time
import re
import os
import glob
import matplotlib.pyplot as plt
from tqdm import tqdm
from together import Together
from collections import defaultdict
import numpy as np

TOGETHER_API_KEY = 'xxx'

# MODEL_NAME = "Qwen/Qwen3-Next-80B-A3B-Thinking"
MODEL_NAME = "deepseek-ai/DeepSeek-R1"

ROOT = "/content/drive/MyDrive/inference-time-scale-adaptation"
CONDITION = "deepseek-r1"

INPUT_DIR = os.path.join(ROOT, "dataset")
OUTPUT_DIR = os.path.join(ROOT, "results", CONDITION)
os.makedirs(OUTPUT_DIR, exist_ok=True)

client = Together(api_key=TOGETHER_API_KEY)


# Helper Functions
def get_model_suffix(full_model_name):
    """
    Parses the model name to create a file suffix.
    Logic: Split by '/', take the last part. Then split by '-', take first two parts.
    Example: "deepseek-ai/DeepSeek-R1" -> "DeepSeek-R1"
    Example: "Qwen/Qwen3-Next-80B" -> "Qwen3-Next"
    """
    try:
        # Step 1: Get the part after the slash (e.g., DeepSeek-R1)
        base_name = full_model_name.split('/')[-1]
        # Step 2: Split by dash and take first two parts
        parts = base_name.split('-')
        if len(parts) >= 2:
            return f"{parts[0]}-{parts[1]}"
        return base_name # Fallback if no dashes
    except Exception:
        return "UnknownModel"

def parse_model_response(response_text):
    """
    Extracts predicted variables from the response.
    Returns a set of integers (e.g., {0, 2}). Returns empty set if failed.
    """
    if not response_text:
        return set()

    # Normalize text
    lower_text = response_text.lower()
    keyword = "final answer:"

    try:
        # Locate the final answer section
        start_idx = lower_text.rindex(keyword)
        answer_part = response_text[start_idx + len(keyword):]

        # Regex to find V0, V1, etc.
        matches = re.findall(r"V(\d+)", answer_part, re.IGNORECASE)

        predicted_indices = set(int(m) for m in matches)
        return predicted_indices

    except ValueError:
        # Keyword not found
        return set()

def get_prediction_with_retry(prompt, model_name, max_retries=5):
    """
    Calls API and attempts to parse.
    Retries if:
    1. API fails (Network error)
    2. Parsing fails (No 'Final Answer' or valid format found)
    """

    for attempt in range(max_retries):
        try:
            # --- 1. API Call ---
            stream = client.chat.completions.create(
                model=model_name,
                temperature=0.5, # Slightly higher temp might help if stuck in a loop
                max_tokens=100000,
                messages=[{"role": "user", "content": prompt}]
            )
            response_content = stream.choices[0].message.content

            # --- 2. Parse Check ---
            predicted_indices = parse_model_response(response_content)

            # If parsing is successful (we got some variables), return immediately
            if predicted_indices:
                return response_content, predicted_indices

            # If parsing failed (empty set), print warning and continue loop (Retry)
            # print(f"   -> [Attempt {attempt+1}/{max_retries}] Parsing failed (Empty result). Retrying...")

        except Exception as e:
            print(f"   -> [Attempt {attempt+1}/{max_retries}] API Error: {e}")
            time.sleep(2) # Wait a bit before retry

    # If all retries fail, return empty
    print(f"   -> ❌ Failed after {max_retries} attempts.")
    return "", set()

def run_batch_pipeline():
    # 1. Get all .pkl files in the directory
    pkl_files = glob.glob(os.path.join(INPUT_DIR, "*.pkl"))

    # Filter out files that look like results (avoid processing already processed files if they are in same dir)
    # 假设原文件不包含 "DeepSeek" 或 "Qwen" 这种模型名，简单过滤防止循环处理
    model_tag = get_model_suffix(MODEL_NAME)
    input_files = [f for f in pkl_files if model_tag not in f]

    print(f"Model Tag: {model_tag}")
    print(f"Found {len(input_files)} datasets to process in {INPUT_DIR}")

    if not input_files:
        print("No input files found. Check your path.")
        return

    # 2. Loop through each file
    for file_path in input_files:
        file_name = os.path.basename(file_path)
        print(f"\n{'='*40}")
        print(f"Processing: {file_name}")
        print(f"{'='*40}")

        # Construct Output Path
        # Logic: input_name.pkl -> input_name_ModelTag.pkl
        name_root, _ = os.path.splitext(file_name)
        output_filename = f"{name_root}_{model_tag}.pkl"
        output_path = os.path.join(OUTPUT_DIR, output_filename)

        try:
            with open(file_path, 'rb') as f:
                dataset = pickle.load(f)
        except Exception as e:
            print(f"Error loading {file_name}: {e}")
            continue

        processed_results = [None] * len(dataset)

        if os.path.exists(output_path):
            try:
                with open(output_path, 'rb') as f:
                    existing_data = pickle.load(f)

                # Fill the placeholder list using indices from the existing file (handles partial runs)
                for idx, item in enumerate(existing_data):
                    if idx < len(processed_results):
                        processed_results[idx] = item
            except Exception as e:
                print(f"⚠️ Exception reading existing file; starting from scratch: {e}")

        # Identify all "invalid" indices (those that are None or have an empty response)
        todo_indices = [
            idx for idx, item in enumerate(processed_results)
            if item is None or not str(item.get('llm_raw_response', '')).strip()
        ]

        if not todo_indices:
            print(f"✅ Skipping {output_filename} (100% complete)")
            continue

        print(f"🔄 {output_filename}: Found {len(todo_indices)} holes to fill out of {len(dataset)} total samples.")

        # Iterate strictly through the todo_indices to "fill the holes"
        pbar = tqdm(todo_indices, desc=f"Filling {name_root[:15]}", unit="sample")

        # This isn't a perfect implementation since we use an additional list to track the "to-do" items.
        # However, given that we typically handle hundreds or thousands of data points, this trade-off is acceptable.

        for i, idx in enumerate(pbar):

            item = dataset[idx]  # Precisely retrieve the original prompt at the target index

            prompt = item['full_prompt']
            ground_truth = set(item['ground_truth'])

            # Call Inference with Retry Logic
            llm_raw_response, predicted_indices = get_prediction_with_retry(
                prompt,
                MODEL_NAME,
                max_retries=5 #
            )

            # Evaluation
            is_correct = (predicted_indices == ground_truth)
            char_count = len(llm_raw_response)

            # Save Results
            result_item = item.copy()
            result_item['llm_raw_response'] = llm_raw_response
            result_item['parsed_prediction'] = predicted_indices
            result_item['is_correct'] = is_correct
            result_item['char_count'] = char_count

            # Optional: Add model metadata
            result_item['model_name'] = MODEL_NAME

            processed_results[idx] = result_item

            if (i + 1) % 20 == 0:
                with open(output_path, 'wb') as f:
                    pickle.dump(processed_results, f)

            # Rate limit buffer
            time.sleep(0.2)

        output_dir_actual = os.path.dirname(output_path)
        if not os.path.exists(output_dir_actual):
            os.makedirs(output_dir_actual, exist_ok=True)
            print(f"📁 Created missing directory: {output_dir_actual}")
        # =======================================================

        # Save to Disk
        try:
            with open(output_path, 'wb') as f:
                pickle.dump(processed_results, f)
            print(f"✅ Saved results to: {output_filename}")
        except Exception as e:
            print(f"❌ Error saving {output_filename}: {e}")
            # 如果再次失败，尝试保存到当前目录作为最后的备份
            backup_name = f"BACKUP_{output_filename}"
            with open(backup_name, 'wb') as f:
                pickle.dump(processed_results, f)
            print(f"⚠️ Saved backup to local colab dir: {backup_name}")

if __name__ == "__main__":
    run_batch_pipeline()

Model Tag: DeepSeek-R1
Found 10 datasets to process in /content/drive/MyDrive/infer_scale_reason/dataset/

Processing: 2_vars_adaptive_sample_5_A_XNOR_B.pkl
✅ Skipping 2_vars_adaptive_sample_5_A_XNOR_B_DeepSeek-R1.pkl (100% complete)

Processing: 2_vars_adaptive_sample_5_A_AND_B.pkl
✅ Skipping 2_vars_adaptive_sample_5_A_AND_B_DeepSeek-R1.pkl (100% complete)

Processing: 2_vars_adaptive_sample_5_A_XOR_B.pkl
✅ Skipping 2_vars_adaptive_sample_5_A_XOR_B_DeepSeek-R1.pkl (100% complete)

Processing: 2_vars_adaptive_sample_5_A_OR_B.pkl
✅ Skipping 2_vars_adaptive_sample_5_A_OR_B_DeepSeek-R1.pkl (100% complete)

Processing: 2_vars_adaptive_sample_5_NOT_(A_AND_B).pkl
✅ Skipping 2_vars_adaptive_sample_5_NOT_(A_AND_B)_DeepSeek-R1.pkl (100% complete)

Processing: 2_vars_adaptive_sample_5_A_AND_(NOT_B).pkl
🔄 2_vars_adaptive_sample_5_A_AND_(NOT_B)_DeepSeek-R1.pkl: Found 82 holes to fill out of 300 total samples.


Filling 2_vars_adaptive:   0%|          | 0/82 [00:00<?, ?sample/s]

   -> [Attempt 1/5] API Error: Error code: 402 - {"message": "Credit limit exceeded. Please navigate to https://api.together.ai/settings/billing to add credit or upgrade your plan.", "type_": "credit_limit"}


Filling 2_vars_adaptive:   1%|          | 1/82 [01:08<1:32:16, 68.36s/sample]

   -> [Attempt 1/5] API Error: Error code: 402 - {"message": "Credit limit exceeded. Please navigate to https://api.together.ai/settings/billing to add credit or upgrade your plan.", "type_": "credit_limit"}


Filling 2_vars_adaptive:   4%|▎         | 3/82 [04:42<2:06:37, 96.17s/sample] 

... (3 outputs left out)

Filling 2_vars_adaptive: 100%|██████████| 300/300 [9:24:08<00:00, 112.83s/sample]


✅ Saved results to: 2_vars_adaptive_sample_5_(NOT_A)_OR_B_DeepSeek-R1.pkl

Processing: 2_vars_adaptive_sample_5_(NOT_A)_AND_(NOT_B).pkl
✅ Skipping 2_vars_adaptive_sample_5_(NOT_A)_AND_(NOT_B)_DeepSeek-R1.pkl (100% complete)

Processing: 2_vars_adaptive_sample_5_(NOT_A)_AND_B.pkl
🔄 2_vars_adaptive_sample_5_(NOT_A)_AND_B_DeepSeek-R1.pkl: Found 20 holes to fill out of 300 total samples.


Filling 2_vars_adaptive:   0%|          | 0/20 [00:03<?, ?sample/s]


KeyboardInterrupt: 